# 从客户快照读懂复购预测

本Notebook用真实数据解释输入、标签和结果，不在这里重复训练所有模型。完整模型运行命令在 docs/06-run-guide.md。下面明确区分当场计算的快照与读取的已保存实验结果。

你需要先运行 `python scripts/extract_data.py`。每次执行从第一格开始，变量只在当前内核中存在。`=` 给变量赋值；函数接收输入并返回结果；`import` 导入工具；带括号是调用函数。

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'data/olist-v2.zip').exists():
    ROOT=ROOT.parent
assert (ROOT/'data/raw').is_dir(), '先在仓库根目录运行 scripts/extract_data.py'
sys.path.insert(0,str(ROOT/'scripts'))
from experiment_common import train,val,test,tm,vm,sm,features
print('已从仓库 data/raw 构建三个历史客户快照。')

COHORTS [{"snapshot": "2017-10-01", "history_start": "2017-04-01", "outcome_end_exclusive": "2017-12-30", "customers": 21160, "repeaters": 211, "prevalence": 0.00997164461247637, "history_orders": 21757}, {"snapshot": "2018-01-01", "history_start": "2017-07-01", "outcome_end_exclusive": "2018-04-01", "customers": 29368, "repeaters": 230, "prevalence": 0.007831653500408608, "history_orders": 30134}, {"snapshot": "2018-04-01", "history_start": "2017-10-01", "outcome_end_exclusive": "2018-06-30", "customers": 37813, "repeaters": 299, "prevalence": 0.007907333456747679, "history_orders": 38715}]


已从仓库 data/raw 构建三个历史客户快照。


## 1. 一个快照，不是一笔交易

下面每行是某个时期的一名客户，包含11个预测输入和未来是否复购的标签y。`.loc` 指定要看的列，`.head(5)`只看前5行。这里为阅读隐藏匿名客户编号，不改变真实数据。

In [2]:
display(train.loc[:,features+['y']].head(5).reset_index(drop=True))

,R,F,M,average_order_value,max_order_value,average_item_rows,freight_share,observed_history_days,recent30_orders,recent30_spend,state,y
0,74.603912,1,13.90,13.90,13.90,1.0,0.520690,74.603912,0.0,0.0,ES,0
1,50.413831,1,76.99,76.99,76.99,1.0,0.156551,50.413831,0.0,0.0,RS,0
2,1.371898,1,35.00,35.00,35.00,1.0,0.252935,1.371898,1.0,35.0,SP,0
3,44.169884,1,229.80,229.80,229.80,2.0,0.107365,44.169884,0.0,0.0,SP,0
4,81.520509,1,299.00,299.00,299.00,1.0,0.082625,81.520509,0.0,0.0,RS,0


R是距最近批准订单天数，F是订单数，M是商品金额。`y=1`表示之后90天有新批准订单，`y=0`表示窗口内没有观察到。`y`必须排除在模型输入之外。

**所以呢：**模型的任务是从左边历史信息学习右边未来标签，而不是读取答案。

## 2. 三个时期的样本数量

`pd.DataFrame` 把几份字典记录变成表。检查训练、验证和测试的正样本数量，比只看总行数更重要。

In [3]:
cohorts=pd.DataFrame([tm,vm,sm])
display(cohorts[['snapshot','history_start','outcome_end_exclusive','customers','repeaters','prevalence']])
assert tm['outcome_end_exclusive'] <= vm['snapshot']
assert vm['outcome_end_exclusive'] <= sm['snapshot']

,snapshot,history_start,outcome_end_exclusive,customers,repeaters,prevalence
0,2017-10-01,2017-04-01,2017-12-30,21160,211,0.009972
1,2018-01-01,2017-07-01,2018-04-01,29368,230,0.007832
2,2018-04-01,2017-10-01,2018-06-30,37813,299,0.007907


`outcome_end_exclusive`表示截止当天不计入。测试只有299名复购者，不是所有37,813人都能提供正样本信息。标签窗口不重叠；历史窗口可以重叠，同一客户可能在不同时间再次评分。

**所以呢：**很高的“全猜不复购”准确率没意义，要看模型能否把稀少的复购者排在前面。

## 3. 读已经完成的模型实验

下面读取 `results/` 中保留的结果，而不是在这格重新训练。`json.loads` 把JSON文字变为字典；循环逐个取模型的测试指标。想重新跑，请按运行指南先执行run_catboost.py，再执行compare_models.py，重跑结果在outputs。

In [4]:
saved=json.loads((ROOT/'results/classification/model-comparison.json').read_text(encoding='utf-8'))
rows=[]
for name,record in saved['models'].items():
    rows.append({'model':name,**record['test']})
metrics=pd.DataFrame(rows)
display(metrics[['model','AP','ROC_AUC','Precision_at_10pct','Recall_at_10pct','selected','hits']])

,model,AP,ROC_AUC,Precision_at_10pct,Recall_at_10pct,selected,hits
0,CatBoost,0.021204,0.603769,0.018509,0.234114,3782,70
1,LogisticRegression,0.021576,0.607284,0.019302,0.244147,3782,73
2,RandomForest,0.022622,0.613442,0.015600,0.197324,3782,59
3,HistGradientBoosting,0.014375,0.582291,0.015071,0.190635,3782,57


前10%名单对每个模型都是3,782人。逻辑回归找到73个复购者，CatBoost找到70个；这一小差异还没有统计显著性的证明。

**所以呢：**先看用途和不确定性，不要自动选最复杂的名字。

## 4. 亲手复算精确率和召回率

下面的数字来自刚才读取的真实结果。精确率的分母是名单人数，召回率的分母是全部实际复购者。`:.2%`把小数显示成两位百分数，计算本身不先舍入。

In [5]:
lr=saved['models']['LogisticRegression']['test']
precision=lr['hits']/lr['selected']
recall=lr['hits']/sm['repeaters']
print(f"名单中复购比例：{lr['hits']}/{lr['selected']} = {precision:.2%}")
print(f"覆盖全部复购者：{lr['hits']}/{sm['repeaters']} = {recall:.2%}")
assert abs(precision-lr['Precision_at_10pct'])<1e-12
assert abs(recall-lr['Recall_at_10pct'])<1e-12

名单中复购比例：73/3782 = 1.93%
覆盖全部复购者：73/299 = 24.41%


## 5. 聚类下一步怎样接进来？

当前分类实验没有使用“组内EDA改进后的特征”。正式研究需要在开发客户上聚类、组内分析、提出候选特征，再做初版与修改版对比。不能把已有分类结果写成“聚类带来的提升”。

## 自测

1. 模型选中73个后来购买的人，能否说营销新增73人？
2. 能否把下一次订单日期加进特征来提高成绩？
3. 如果分类指标不提高，聚类是否必然没价值？

<details><summary>参考答案</summary>

1. 不能，没有营销处理组与对照组。这些人可能原本就会买。
2. 不能，预测时尚未知，属于信息泄漏。
3. 不必然。稳定、可解释且与决策相关的群体差异可以独立提供洞察，但不能仅凭群名称证明价值。

</details>